# 06. SQL 에이전트 심화 -- LangChain & LangGraph

자연어를 SQL 쿼리로 변환하는 에이전트를 두 가지 방법으로 구축합니다: LangChain `create_agent` + `SQLDatabaseToolkit` (간단 버전)과 LangGraph `StateGraph` (커스텀 버전). Human-in-the-Loop, `interrupt()`, `Command(resume=...)` 패턴을 다룹니다.

## 학습 목표

- SQL 에이전트의 8단계 워크플로우를 이해한다
- `SQLDatabase`와 `SQLDatabaseToolkit`의 4개 도구를 활용한다
- LangChain `create_agent`로 ReAct 기반 SQL Agent를 구현한다
- `HumanInTheLoopMiddleware`로 쿼리 실행 전 승인을 추가한다
- LangGraph `StateGraph`로 커스텀 SQL Agent를 구축한다
- `bind_tools`와 `tool_choice`로 강제 도구 호출을 설정한다
- `interrupt()`와 `Command(resume=...)`로 쿼리 리뷰를 구현한다

## 6.1 환경 설정 (SQLite + Chinook DB)

In [ ]:
# %pip install langchain langchain-openai langchain-community langgraph sqlalchemy python-dotenv

from dotenv import load_dotenv

load_dotenv(override=True)

from langchain_openai import ChatOpenAI
from langchain_community.utilities import SQLDatabase

llm = ChatOpenAI(model="gpt-4.1")
db = SQLDatabase.from_uri("sqlite:///Chinook.db")
print(f"Dialect: {db.dialect}")

In [ ]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler

    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


## 6.2 SQL 에이전트 개요

SQL 에이전트는 자연어 질문을 SQL 쿼리로 변환하는 **8단계** 프로세스를 따릅니다:

```
1. 질문 수신 -> 2. 테이블 목록 -> 3. 관련 테이블 스키마
-> 4. SQL 쿼리 생성 -> 5. 쿼리 검증 -> 6. (선택) 사람 리뷰
-> 7. 쿼리 실행 -> 8. 결과 해석
```

### 왜 에이전트가 필요한가?

단순 text-to-SQL과 달리 에이전트 방식은 **스키마 탐색 → 쿼리 생성 → 검증 → 실행**의 반복 루프를 수행합니다. 잘못된 쿼리가 생성되면 에이전트가 오류를 분석하고 쿼리를 재작성할 수 있어 정확도가 크게 향상됩니다. 또한 에이전트는 필요한 테이블의 스키마만 선택적으로 로드하므로 **컨텍스트 윈도우를 효율적으로** 사용합니다.

### 에이전트 실행 트레이스 예시

```
User: "지난달 매출 상위 5개 제품은?"

Agent -> sql_db_list_tables()
      <- "customers, orders, order_items, products, categories"

Agent -> sql_db_schema("orders, order_items, products")
      <- CREATE TABLE orders (id INT, order_date DATE, ...)
         CREATE TABLE order_items (order_id INT, product_id INT, quantity INT, price DECIMAL, ...)

Agent -> sql_db_query_checker("SELECT p.name, SUM(oi.quantity * oi.price) ...")
      <- "The query looks correct."

Agent -> sql_db_query(validated_query)
      <- [("Widget Pro", 45230.00), ("Gadget X", 38100.00), ...]
```

### 안전 수칙

| 우려사항 | 대응 |
|---|---|
| SQL Injection | 파라미터화된 쿼리 사용, Toolkit이 자동 처리 |
| DML 실행 | 시스템 프롬프트에서 INSERT/UPDATE/DELETE 금지, DB 레벨 읽기 전용 권한 설정 |
| 비용 높은 쿼리 | LIMIT 강제, Human-in-the-Loop으로 실행 전 승인 |
| 민감 데이터 | `include_tables`/`exclude_tables`로 접근 가능 테이블 제한, 컬럼 레벨 권한 설정 |
| 데이터 노출 | 데이터베이스 뷰(view) 또는 제한된 사용자 권한 활용 |

### 접근 가능 테이블 제한

프로덕션에서는 에이전트가 접근할 수 있는 테이블을 명시적으로 제한하는 것이 좋습니다:

```python
db = SQLDatabase.from_uri(
    "sqlite:///company.db",
    include_tables=["products", "orders", "order_items"],  # 허용 목록
    # exclude_tables=["users", "credentials"],             # 또는 차단 목록
)
```

## 6.3 SQLDatabaseToolkit

4개의 도구를 자동 생성합니다:

| 도구 | 기능 |
|---|---|
| `sql_db_list_tables` | 데이터베이스의 모든 테이블 이름 반환 |
| `sql_db_schema` | CREATE TABLE 문 + 샘플 행 반환 |
| `sql_db_query` | SQL 쿼리를 실행하고 결과 반환 |
| `sql_db_query_checker` | LLM이 쿼리의 오류를 사전 검사 |

In [ ]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)
tools = toolkit.get_tools()

for t in tools:
    print(f"  {t.name}: {t.description[:60]}...")
print(f"총 도구 수: {len(tools)}")

## 6.4 LangChain SQL Agent -- `create_agent` + ReAct

`create_agent`는 LangChain의 고수준 API로, 모델과 도구를 받아 **ReAct(Reasoning + Acting) 루프**를 자동으로 구성합니다. 에이전트는 시스템 프롬프트에 정의된 워크플로우를 따라 도구를 순서대로 호출합니다.

### ReAct 루프 동작 원리

1. LLM이 사용자 질문과 대화 이력을 분석하여 **다음에 호출할 도구**를 결정합니다
2. 도구가 실행되고 결과가 대화 이력에 추가됩니다
3. LLM이 결과를 확인하고, 추가 도구 호출이 필요하면 1단계로 돌아갑니다
4. 최종 답변이 준비되면 텍스트 응답을 반환합니다

### 시스템 프롬프트의 역할

시스템 프롬프트는 에이전트의 행동 지침을 정의합니다. SQL 에이전트에서는 특히 다음을 명시해야 합니다:
- **도구 호출 순서**: `list_tables` → `schema` → `query_checker` → `query` 순서 강제
- **안전 규칙**: `LIMIT` 사용, DML 금지, 필요한 컬럼만 조회
- **오류 처리**: 쿼리 오류 발생 시 재작성 지시
- **SQL 방언**: 현재 DB의 dialect(SQLite, PostgreSQL 등) 명시

In [ ]:
system_prompt = (
    "당신은 SQL 에이전트입니다. 단계:\n"
    "1. sql_db_list_tables\n2. sql_db_schema\n"
    "3. 쿼리 작성 + sql_db_query_checker\n"
    "4. sql_db_query\n5. 결과를 해석하세요.\n"
    f"규칙: LIMIT 10 사용. DML 금지. Dialect: {db.dialect}"
)

In [ ]:
from langchain.agents import create_agent

sql_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
)
print("LangChain SQL 에이전트 생성됨.")

## 6.5 실행 테스트

In [ ]:
response = sql_agent.invoke(
    {"messages": [{"role": "user", "content": "어떤 나라의 고객이 가장 많나요?"}]},
    config=lf_config,
)
print(response["messages"][-1].content)

## 6.6 HITL -- `HumanInTheLoopMiddleware`

프로덕션 환경에서는 SQL 쿼리 실행 전 **사람의 승인이 필수**입니다. 에이전트가 생성한 쿼리가 비용이 높거나, 예상치 못한 테이블에 접근하거나, 의도와 다른 결과를 반환할 수 있기 때문입니다.

`HumanInTheLoopMiddleware`는 지정된 도구(`sql_db_query`) 호출을 가로채서 실행을 일시 중단하고, 사람이 리뷰할 수 있도록 합니다.

### 리뷰 옵션 3가지

에이전트가 `sql_db_query`를 호출하려 할 때, 실행이 일시 중단되고 사람은 다음 중 하나를 선택합니다:

| 옵션 | `Command(resume=...)` 값 | 설명 |
|------|--------------------------|------|
| **승인** | `"approve"` | 생성된 쿼리를 그대로 실행 |
| **수정** | `{"type": "edit", "args": {"query": "..."}}` | 쿼리를 수정한 후 실행 |
| **거부** | `{"type": "reject", "reason": "..."}` | 쿼리를 실행하지 않고 사유를 전달 |

### 왜 HITL이 중요한가?

- **비용 제어**: `LIMIT` 없는 대규모 테이블 풀 스캔 방지
- **데이터 보호**: 민감한 컬럼 접근 사전 차단
- **정확성 검증**: 에이전트가 질문 의도를 잘못 해석한 경우 수정 가능
- **감사 추적(Audit Trail)**: 모든 실행된 쿼리에 대한 승인 기록 유지

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

hitl = HumanInTheLoopMiddleware(
    interrupt_on={"sql_db_query": True},
)
sql_agent_hitl = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt,
    middleware=[hitl],
)
print("HITL이 적용된 SQL 에이전트 생성됨.")

In [ ]:
from langgraph.types import Command

# Option 1: Approve
# result = sql_agent_hitl.invoke(Command(resume="approve"), config=lf_config)

# Option 2: Edit query
# result = sql_agent_hitl.invoke(Command(resume={
#     "type": "edit",
#     "args": {"query": "SELECT Country, COUNT(*) ..."}
# }), config=lf_config)
print("HITL 재개 옵션: approve / edit / reject")

## 6.7 LangGraph 커스텀 SQL Agent -- StateGraph

LangChain `create_agent`는 빠르게 프로토타입을 만들 수 있지만, **노드 단위의 세밀한 제어**가 필요하면 LangGraph `StateGraph`를 사용합니다. 각 단계를 독립적인 노드로 정의하여 다음을 실현할 수 있습니다:

- **조건부 분기**: 쿼리 검증 실패 시 재생성 노드로 라우팅
- **강제 도구 호출**: `bind_tools(tool_choice=...)`로 특정 노드에서 반드시 특정 도구 호출
- **세밀한 중단점**: `interrupt()`로 원하는 노드에서 정확히 실행 중단
- **커스텀 상태**: 쿼리 이력, 재시도 횟수 등을 상태에 추가

### 그래프 구조

```
START -> list_tables -> get_schema -> generate_query
      -> check_query -> execute_query -> END
```

각 노드는 공유 `State` 객체를 받아 메시지를 추가하며, 에이전트가 워크플로우를 진행하는 동안 대화 이력이 누적됩니다. `tools_condition`을 사용하면 `check_query` 결과에 따라 쿼리를 재생성하거나 실행으로 진행하는 조건부 분기를 구현할 수 있습니다.

### LangChain `create_agent` 대비 장점

| 측면 | `create_agent` | `StateGraph` |
|------|---------------|--------------|
| 도구 호출 순서 | LLM 자율 결정 | 그래프 엣지로 강제 |
| 오류 시 재시도 | 시스템 프롬프트에 의존 | 조건부 엣지로 명시적 구현 |
| 사람 리뷰 | 미들웨어 기반 | `interrupt()` 기반, 위치 자유 |
| 디버깅 | 블랙박스 | 노드별 상태 확인 가능 |

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages


class SQLState(TypedDict):
    messages: Annotated[list, add_messages]


print(f"SQLState 키: {list(SQLState.__annotations__)}")

## 6.8 전용 노드 -- `list_tables`, `get_schema`, `generate_query`, `check_query`

각 노드는 SQL 에이전트 워크플로우의 한 단계를 담당합니다.

In [ ]:
tool_map = {t.name: t for t in tools}

list_tbl = tool_map["sql_db_list_tables"]
schema_tl = tool_map["sql_db_schema"]
query_tl = tool_map["sql_db_query"]
check_tl = tool_map["sql_db_query_checker"]


def list_tables_node(state: SQLState):
    tables = list_tbl.invoke("", config=lf_config)

    msg = {"role": "assistant", "content": f"테이블: {tables}"}

    return {"messages": [msg]}

In [ ]:
def get_schema_node(state: SQLState):
    """관련 테이블의 스키마를 가져옵니다."""
    resp = llm.invoke(
        state["messages"]
        + [
            {
                "role": "user",
                "content": "관련 테이블이 어떤 것인가요? 이름만 알려주세요.",
            }
        ],
        config=lf_config,
    )
    schema = schema_tl.invoke(resp.content.strip(), config=lf_config)
    msg = {"role": "assistant", "content": f"스키마:\n{schema}"}
    return {"messages": [msg]}

## 6.9 `bind_tools` with `tool_choice` -- 강제 도구 호출

`tool_choice` 파라미터로 특정 도구를 **강제** 호출하도록 설정합니다.

In [ ]:
llm_forced = llm.bind_tools([check_tl], tool_choice="sql_db_query_checker")


def generate_query_node(state: SQLState):
    """SQL 쿼리를 생성합니다."""
    prompt = "SQL 쿼리를 작성해주세요. checker 도구를 사용하세요."
    msgs = state["messages"] + [{"role": "user", "content": prompt}]
    response = llm_forced.invoke(msgs, config=lf_config)
    return {"messages": [response]}

In [ ]:
def check_query_node(state: SQLState):
    """
    생성된 쿼리를 검증합니다.
    """

    last = state["messages"][-1]

    if hasattr(last, "tool_calls") and last.tool_calls:
        query = last.tool_calls[0]["args"].get("query", "")

        result = check_tl.invoke(query, config=lf_config)

        return {"messages": [{"role": "tool", "content": result}]}

    return state

## 6.10 `interrupt()`로 쿼리 리뷰

LangGraph의 `interrupt()` 함수는 그래프 실행을 **일시 중단**하고 외부 입력(사람의 리뷰)을 기다립니다. `HumanInTheLoopMiddleware`와 달리 `interrupt()`는 **노드 내부 코드의 정확한 위치**에서 중단할 수 있어 더 유연합니다.

### 동작 원리

1. 노드 함수 내에서 `interrupt(payload)`를 호출하면 그래프 실행이 즉시 중단됩니다
2. `payload`는 클라이언트에게 전달되어 리뷰 UI에 표시됩니다 (예: 생성된 SQL 쿼리)
3. 클라이언트가 `Command(resume=value)`로 그래프를 재개하면, `interrupt()`가 `value`를 반환합니다
4. 노드 함수는 반환된 값에 따라 쿼리를 실행, 수정, 또는 거부합니다

### `interrupt()` vs `HumanInTheLoopMiddleware`

| 특성 | `interrupt()` | `HumanInTheLoopMiddleware` |
|------|--------------|---------------------------|
| 적용 범위 | 노드 내 코드 레벨 | 도구 호출 레벨 |
| 유연성 | 임의 로직 구현 가능 | 도구 호출 가로채기만 가능 |
| 상태 접근 | 전체 State 접근 가능 | 도구 인자만 접근 가능 |
| 체크포인터 | 필수 (상태 저장 필요) | 선택적 |

In [ ]:
from langgraph.types import interrupt


def execute_query_node(state: SQLState):
    """
    사람의 리뷰 후 쿼리를 실행합니다.
    """

    query = state["messages"][-1].content

    review = interrupt({"query": query, "action": "review_sql"})

    if review.get("action") == "accept":
        result = query_tl.invoke(query, config=lf_config)

    elif review.get("action") == "edit":
        result = query_tl.invoke(review["edited_query"], config=lf_config)

    else:
        result = f"거부됨: {review.get('reason', '')}"

    return {"messages": [{"role": "assistant", "content": result}]}

## 6.11 `Command(resume=...)` 패턴

`interrupt()`로 중단된 그래프를 재개하려면 `Command(resume=...)`를 사용합니다.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

builder = StateGraph(SQLState)
builder.add_node("list_tables", list_tables_node)
builder.add_node("get_schema", get_schema_node)
builder.add_node("generate_query", generate_query_node)
builder.add_node("check_query", check_query_node)
builder.add_node("execute_query", execute_query_node)

In [ ]:
builder.add_edge(START, "list_tables")
builder.add_edge("list_tables", "get_schema")
builder.add_edge("get_schema", "generate_query")
builder.add_edge("generate_query", "check_query")
builder.add_edge("check_query", "execute_query")
builder.add_edge("execute_query", END)

checkpointer = InMemorySaver()
sql_graph = builder.compile(checkpointer=checkpointer)
print("LangGraph SQL 에이전트 컴파일됨.")

In [ ]:
from langgraph.types import Command

config = {"configurable": {"thread_id": "sql-1"}}

# Resume examples after interrupt:
# sql_graph.invoke(Command(resume={"action": "accept"}), {**config, **lf_config})
# sql_graph.invoke(Command(resume={
#     "action": "edit", "edited_query": "SELECT ..."
# }), {**config, **lf_config})
print("Command(resume=...) 패턴: accept / edit / reject")

## 요약

### 두 가지 SQL Agent 비교

| 특성 | LangChain `create_agent` | LangGraph `StateGraph` |
|---|---|---|
| 구현 복잡도 | 낮음 (5줄) | 높음 (전용 노드) |
| 제어 수준 | ReAct 자동 | 노드 단위 커스텀 |
| HITL | `HumanInTheLoopMiddleware` | `interrupt()` + `Command(resume=...)` |
| 강제 도구 호출 | 미지원 | `bind_tools(tool_choice=...)` |
| 적합한 경우 | 빠른 프로토타입 | 프로덕션, 세밀한 제어 |

### HITL 패턴

| 액션 | `Command(resume=...)` |
|---|---|
| Accept | `{"action": "accept"}` |
| Edit | `{"action": "edit", "edited_query": "..."}` |
| Reject | `{"action": "reject", "reason": "..."}` |

### SQLDatabaseToolkit 4개 도구

| 도구 | 단계 | 용도 |
|---|---|---|
| `sql_db_list_tables` | 2 | 테이블 목록 확인 |
| `sql_db_schema` | 3 | DDL + 샘플 데이터 조회 |
| `sql_db_query_checker` | 5 | 쿼리 사전 검증 |
| `sql_db_query` | 7 | 쿼리 실행 |

### 다음 단계
→ **[07_data_analysis.ipynb](./07_data_analysis.ipynb)**: 데이터 분석 에이전트를 만듭니다.
